# Exercise rejuvenation (aged mouse brain MERFISH): s carries the clock shift, z does not

Old control (4 mice) vs old exercise (4 mice), 594,896 cells x 300 genes (Sun et al. aging/exercise atlas).
Ground truth: the published Spatial Aging Clock applied to observed expression (never seen by Haruka).
Out-of-fold ridge predictions of clock age from z or from s (train 3+3 mice, predict the held-out pair) per
cell type x brain region: s reproduces the observed exercise shift (r = +0.85 across strata, rejuvenation strongest in
vascular cells: VLMC, VSMC, endothelium), z does not (r = -0.17).

In [ ]:
import sys, pathlib; sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), pathlib.Path.cwd() / 'notebooks'] if (p / 'repro.py').exists())))
from repro import run, require, show, fetch, fetch_fitted, STAGES, H, WK, PY, PY_VIO, GPU
# Default: download inputs + the paper's fitted Haruka outputs from Hugging Face, then analyses -> figure.
# The Haruka fit step reuses the unpacked fit; HARUKA_REFIT=1 (env) refits from scratch. Auxiliary fits (baselines, seeds) still run.
# Redraw only from saved outputs: STAGES.update(download=False, fit=False, analysis=False)  (or env HARUKA_FIGURE_ONLY=1)
# Rebuild the inputs from the raw public releases instead of Hugging Face: STAGES.update(data=True)
print(H, STAGES)

## 0 Preprocessed data (Hugging Face)

The ready-to-fit inputs are read directly from [C1vy/haruka_v2_data](https://huggingface.co/datasets/C1vy/haruka_v2_data) (cached in `hf_data/`) and linked to the paths the scripts read. The data stage below rebuilds the same files from the raw releases and is off by default.

In [ ]:
fetch('04_exercise/exercise_old_full.h5ad', WK / 'haruka_exercise/data/exercise_old_full.h5ad')

The paper's default Haruka fit, published as fitted AnnData (input + z, s, predicted neighbourhoods; see the dataset card). It is unpacked into the fit folder, so the fit step below reuses it. Set `HARUKA_REFIT=1` before launching to refit instead.

In [ ]:
fetch_fitted('fitted/04_exercise/exercise_old_full_haruka_fitted.h5ad', WK / 'haruka_exercise/results_w10_s100/exercise_old_full/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0')

## 1 Data: keep old control / old exercise mice

In [ ]:
run(f'{PY} bench_exercise/prepare_exercise.py', 'data')  # /home/yancui/virtualorganoid/exercise.h5ad -> exercise_old_full.h5ad

## 2 Fit (default model; batch = run; ~15 min on one GPU)

In [ ]:
run(f'bash bench_exercise/run_fit_w10_s100.sh {GPU}', 'fit', log='bench_exercise/logs/fit_w10_s100.log')
require(WK / 'haruka_exercise/results_w10_s100/exercise_old_full/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/background.npy', WK / 'haruka_exercise/results_w10_s100/exercise_old_full/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0/salient.npy')

## 3 Spatial Aging Clock on observed expression (github sunericd/SpatialAgingClock @ 2b832f4)

In [ ]:
CLK = 'bench_oskm_reprogramming/external/SpatialAgingClock'
run(f'[ -d {CLK} ] || (git clone -q https://github.com/sunericd/SpatialAgingClock {CLK} && git -C {CLK} checkout -q 2b832f4)', 'download')
run(f'{PY} bench_exercise/run_clock_exercise.py', 'analysis', log='bench_exercise/clock.log')

## 4 Held-out clock prediction from z and s; within-vascular partition; panels
`make_exercise_clock_shift.py` reuses `oof_clock_predictions_w10_s100.csv.gz` if present: move it away for a true rerun.

In [ ]:
E = H / 'figures_paper/exercise'
run(f'{PY} make_exercise_clock_shift.py', 'figure', cwd=E)
run(f'{PY} intra_vascular.py', 'analysis', cwd=E)
run(f'{PY} make_intra_panel.py', 'figure', cwd=E)
run(f'{PY} make_exercise_blocks.py', 'figure', cwd=E)
run(f'{PY} ../compose_montage.py exercise_clock && {PY} ../compose_montage.py exercise_blocks', 'figure', cwd=E)

In [ ]:
import pandas as pd
d = pd.read_csv(E / 'exercise_clock_shift_by_celltype_region.csv')
print('r(observed, z) = %.2f   r(observed, s) = %.2f' % (d.observed.corr(d.z), d.observed.corr(d.s)))

## Figure

In [ ]:
show(H / 'figures_paper/exercise/Exercise_clock_shift_full.png')

In [ ]:
show(H / 'figures_paper/exercise/Exercise_blocks_full.png')